In [ ]:

# Ejecutar solo si Java no está instalado.

# !apt-get update
# !apt-get install -y openjdk-11-jdk


In [ ]:

# Instalar dependencias Python
!pip install -q python-terrier beautifulsoup4 lxml


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00


In [ ]:

import os
import re
import tarfile
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

import pyterrier as pt

if not pt.java.started():
    pt.java.init()

print("PyTerrier inicializado correctamente")


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done
PyTerrier inicializado correctamente


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


#### Descarga de Wiki-small y Parseo HTML

In [ ]:
# Extrae wiki-small del link adjuntado en el TP

COLLECTION_URL = "http://dg3rtljvitrle.cloudfront.net/wiki-small.tar.gz"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

archive_path = data_dir / "wiki-small.tar.gz"
extract_dir = data_dir / "wiki-small"

if not archive_path.exists():
    !wget -O {archive_path} {COLLECTION_URL}

if extract_dir.exists():
    shutil.rmtree(extract_dir)

extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(extract_dir)

print("Colección descomprimida en:", extract_dir)
print("Cantidad de archivos encontrados:", len(list(extract_dir.rglob("*"))))


# Son todos elementos HTML. Se procede a parsear los documentos, eliminando tags.

def clean_html_file(path: Path) -> dict:
    """Lee un archivo HTML y devuelve un diccionario apto para PyTerrier."""
    raw = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "lxml")

    # Eliminar contenido no textual o poco útil para recuperación
    for tag in soup(["script", "style", "noscript", "meta", "link"]):
        tag.decompose()

    # Título
    if soup.title and soup.title.get_text(strip=True):
        title = soup.title.get_text(" ", strip=True)
    else:
        h1 = soup.find("h1")
        title = h1.get_text(" ", strip=True) if h1 else path.stem

    # Texto visible
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()

    # docno estable y corto
    relative = path.relative_to(extract_dir).as_posix()
    docno = re.sub(r"[^A-Za-z0-9_\-]+", "_", relative)
    if len(docno) > 120:
        docno = docno[-120:]

    return {
        "docno": docno,
        "title": title[:250],
        "filename": relative[:500],
        "text": text
    }


# Busca archivos HTML
html_files = [
    p for p in extract_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {".html", ".htm"}
]

print("Archivos HTML encontrados:", len(html_files))

docs = [clean_html_file(p) for p in html_files]
df_docs = pd.DataFrame(docs)

df_docs.head()


Archivos HTML encontrados: 6043
Documentos parseados: 6043


,docno,title,filename,text
0,en_articles_g_d_p_GDP_density_fc55_html,"GDP density - Wikipedia, the free encyclopedia",en/articles/g/d/p/GDP_density_fc55.html,"GDP density - Wikipedia, the free encyclopedia..."
1,en_articles_g_a_t_Gate_House_f817_html,"Gate House - Wikipedia, the free encyclopedia",en/articles/g/a/t/Gate_House_f817.html,"Gate House - Wikipedia, the free encyclopedia ..."
2,en_articles_g_a_t_Gateway_Grizzlies_1169_html,"Gateway Grizzlies - Wikipedia, the free encycl...",en/articles/g/a/t/Gateway_Grizzlies_1169.html,"Gateway Grizzlies - Wikipedia, the free encycl..."
3,en_articles_g_a_b_Gabrias_html,"Gabrias - Wikipedia, the free encyclopedia",en/articles/g/a/b/Gabrias.html,"Gabrias - Wikipedia, the free encyclopedia Gab..."
4,en_articles_g_a_b_Gabriel_of_Comane_dde7_html,"Gabriel of Comane - Wikipedia, the free encycl...",en/articles/g/a/b/Gabriel_of_Comane_dde7.html,"Gabriel of Comane - Wikipedia, the free encycl..."


#### Indexación con PyTerrier


In [ ]:
# La columna `text` es el contenido indexable.
#`title` y `filename` se guardan como metadata para visualizar resultados.

index_dir = Path("/content/wiki_small_index")

if index_dir.exists():
    shutil.rmtree(index_dir)

index_dir.mkdir(parents=True, exist_ok=True)

indexer = pt.IterDictIndexer(
    str(index_dir),
    overwrite=True,
    meta={
        "docno": 128,
        "title": 256,
        "filename": 512
    }
)

indexref = indexer.index(df_docs[["docno", "title", "filename", "text"]].to_dict(orient="records"))

print("Índice creado en:", indexref)


Índice creado en: <org.terrier.querying.IndexRef at 0x79737a374190 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x2d41fcd8 at 0x797379f2aff0>>


In [ ]:
# Estadisticas de indexado
index = pt.IndexFactory.of(indexref)
print(index.getCollectionStatistics().toString())


Number of documents: 6043
Number of terms: 164927
Number of postings: 1563535
Number of fields: 0
Number of tokens: 2870991
Field names: []
Positions:   false



#### Queries manuales. Recuperación con TF-IDF y BM25


In [ ]:
# Armado de queries apartir de 5 necesidades de informacion
topics = pd.DataFrame([
    {
        "qid": "1",
        "need": "Información del Lenguaje de programación Python",
        "query": "python programming language"
    },
    {
        "qid": "2",
        "need": "Paises que participaron en la Segunda Guerra Mundial",
        "query": "world war ii countries"
    },
    {
        "qid": "3",
        "need": "Plantas que existen en Argentina",
        "query": "plants Argentina"
    },
    {
        "qid": "4",
        "need": "Libros de ciencia ficcón",
        "query": "science fiction books"
    },
    {
        "qid": "5",
        "need": "Población (cantidad) de Nagasaki, Japón",
        "query": "population nagasaki japan"
    },
])

# Prueba la recuperacion con metodos TF*IDF y BM25
tfidf = pt.terrier.Retriever(
    index,
    wmodel="TF_IDF",
    metadata=["docno", "title", "filename"],
    num_results=50
)

bm25 = pt.terrier.Retriever(
    index,
    wmodel="BM25",
    metadata=["docno", "title", "filename"],
    num_results=50
)

tfidf_results = tfidf.transform(topics)
bm25_results = bm25.transform(topics)

tfidf_results.head()



Resultados TF-IDF: (250, 9)
Resultados BM25: (250, 9)


,qid,docid,docno,title,filename,rank,score,need,query
0,1,2930,en_articles_r_o_x_ROX_Desktop_b4f8_html,"ROX Desktop - Wikipedia, the free encyclopedia",en/articles/r/o/x/ROX_Desktop_b4f8.html,0,9.538912,Información del Lenguaje de programación Python,python programming language
1,1,4466,en_articles_c_l_a_Class__computer_science_html,"Class (computer science) - Wikipedia, the free...",en/articles/c/l/a/Class_(computer_science).html,1,8.017001,Información del Lenguaje de programación Python,python programming language
2,1,256,en_articles_g_u_o_Guo_Youzhi_8db1_html,"Guo Youzhi - Wikipedia, the free encyclopedia",en/articles/g/u/o/Guo_Youzhi_8db1.html,2,7.415881,Información del Lenguaje de programación Python,python programming language
3,1,2532,en_articles_h_a_n_Hans_Moleman_c002_html,"Hans Moleman - Wikipedia, the free encyclopedia",en/articles/h/a/n/Hans_Moleman_c002.html,3,6.313163,Información del Lenguaje de programación Python,python programming language
4,1,5478,en_articles_l_o_u_Lout_html,"Lout - Wikipedia, the free encyclopedia",en/articles/l/o/u/Lout.html,4,6.172097,Información del Lenguaje de programación Python,python programming language


#### Ranking finales (Top 50)

In [ ]:
# Función que muestra el ranking de cada medoto de recuperación para la query indicada.
def show_ranking(qid: str, top_k: int = 10):
    query_text = topics.loc[topics["qid"] == qid, "query"].iloc[0]
    need_text = topics.loc[topics["qid"] == qid, "need"].iloc[0]

    print(f"QID {qid}")
    print("Necesidad:", need_text)
    print("Query:", query_text)
    print()

    cols = ["rank", "docno", "score", "title", "filename"]

    tfidf_top = (
        tfidf_results[tfidf_results["qid"] == qid]
        .sort_values("rank")
        .head(top_k)[cols]
        .reset_index(drop=True)
    )

    bm25_top = (
        bm25_results[bm25_results["qid"] == qid]
        .sort_values("rank")
        .head(top_k)[cols]
        .reset_index(drop=True)
    )

    print("=== TF-IDF ===")
    display(tfidf_top)

    print("=== BM25 ===")
    display(bm25_top)

# Se mostrara solo la primera Q a modo de ejemplo.
show_ranking("1", top_k=50)


QID 1
Necesidad: Información del Lenguaje de programación Python
Query: python programming language

=== TF-IDF ===


,rank,docno,score,title,filename
0,0,en_articles_r_o_x_ROX_Desktop_b4f8_html,9.538912,"ROX Desktop - Wikipedia, the free encyclopedia",en/articles/r/o/x/ROX_Desktop_b4f8.html
1,1,en_articles_c_l_a_Class__computer_science_html,8.017001,"Class (computer science) - Wikipedia, the free...",en/articles/c/l/a/Class_(computer_science).html
2,2,en_articles_g_u_o_Guo_Youzhi_8db1_html,7.415881,"Guo Youzhi - Wikipedia, the free encyclopedia",en/articles/g/u/o/Guo_Youzhi_8db1.html
3,3,en_articles_h_a_n_Hans_Moleman_c002_html,6.313163,"Hans Moleman - Wikipedia, the free encyclopedia",en/articles/h/a/n/Hans_Moleman_c002.html
4,4,en_articles_l_o_u_Lout_html,6.172097,"Lout - Wikipedia, the free encyclopedia",en/articles/l/o/u/Lout.html
5,5,en_articles_w_i_l_Wilpattu_National_Park_cf60_...,6.031236,"Wilpattu National Park - Wikipedia, the free e...",en/articles/w/i/l/Wilpattu_National_Park_cf60....
6,6,en_articles_w_i_n_Winter_solstice__disambiguat...,5.902099,"Winter solstice (disambiguation) - Wikipedia, ...",en/articles/w/i/n/Winter_solstice_(disambiguat...
7,7,en_articles_s_i_n_SinoVision_32c9_html,5.869495,"SinoVision - Wikipedia, the free encyclopedia",en/articles/s/i/n/SinoVision_32c9.html
8,8,en_articles_s_t_e_Stellarium_html,5.575250,"Stellarium - Wikipedia, the free encyclopedia",en/articles/s/t/e/Stellarium.html
9,9,en_articles_j_f_i_JFin_1830_html,5.501144,"JFin - Wikipedia, the free encyclopedia",en/articles/j/f/i/JFin_1830.html


=== BM25 ===


,rank,docno,score,title,filename
0,0,en_articles_r_o_x_ROX_Desktop_b4f8_html,15.582171,"ROX Desktop - Wikipedia, the free encyclopedia",en/articles/r/o/x/ROX_Desktop_b4f8.html
1,1,en_articles_g_u_o_Guo_Youzhi_8db1_html,11.657097,"Guo Youzhi - Wikipedia, the free encyclopedia",en/articles/g/u/o/Guo_Youzhi_8db1.html
2,2,en_articles_c_l_a_Class__computer_science_html,11.552309,"Class (computer science) - Wikipedia, the free...",en/articles/c/l/a/Class_(computer_science).html
3,3,en_articles_w_i_l_Wilpattu_National_Park_cf60_...,10.990245,"Wilpattu National Park - Wikipedia, the free e...",en/articles/w/i/l/Wilpattu_National_Park_cf60....
4,4,en_articles_w_i_n_Winter_solstice__disambiguat...,10.754929,"Winter solstice (disambiguation) - Wikipedia, ...",en/articles/w/i/n/Winter_solstice_(disambiguat...
5,5,en_articles_h_a_n_Hans_Moleman_c002_html,10.545414,"Hans Moleman - Wikipedia, the free encyclopedia",en/articles/h/a/n/Hans_Moleman_c002.html
6,6,en_articles_j_f_i_JFin_1830_html,10.024300,"JFin - Wikipedia, the free encyclopedia",en/articles/j/f/i/JFin_1830.html
7,7,en_articles_l_i_s_List_of_TV_shows_made_into_f...,9.252253,"List of TV shows made into films - Wikipedia, ...",en/articles/l/i/s/List_of_TV_shows_made_into_f...
8,8,en_articles_t_r_a_Trade_Descriptions_Act_1968_...,9.246166,"Trade Descriptions Act 1968 - Wikipedia, the f...",en/articles/t/r/a/Trade_Descriptions_Act_1968_...
9,9,en_articles_l_i_s_List_of_Arrow_Dynamics_rolle...,8.752429,List of Arrow Dynamics roller coasters - Wikip...,en/articles/l/i/s/List_of_Arrow_Dynamics_rolle...


#### Exportar resultados

In [ ]:

tfidf_export = tfidf_results.copy()
tfidf_export["model"] = "TF_IDF"

bm25_export = bm25_results.copy()
bm25_export["model"] = "BM25"

rankings_export = pd.concat([tfidf_export, bm25_export], ignore_index=True)
rankings_export = rankings_export[[
    "model", "qid", "query", "rank", "docno", "score", "title", "filename"
]]

rankings_export.to_csv("wiki_small_rankings_top50.csv", index=False)

rankings_export.head()


,model,qid,query,rank,docno,score,title,filename
0,TF_IDF,1,python programming language,0,en_articles_r_o_x_ROX_Desktop_b4f8_html,9.538912,"ROX Desktop - Wikipedia, the free encyclopedia",en/articles/r/o/x/ROX_Desktop_b4f8.html
1,TF_IDF,1,python programming language,1,en_articles_c_l_a_Class__computer_science_html,8.017001,"Class (computer science) - Wikipedia, the free...",en/articles/c/l/a/Class_(computer_science).html
2,TF_IDF,1,python programming language,2,en_articles_g_u_o_Guo_Youzhi_8db1_html,7.415881,"Guo Youzhi - Wikipedia, the free encyclopedia",en/articles/g/u/o/Guo_Youzhi_8db1.html
3,TF_IDF,1,python programming language,3,en_articles_h_a_n_Hans_Moleman_c002_html,6.313163,"Hans Moleman - Wikipedia, the free encyclopedia",en/articles/h/a/n/Hans_Moleman_c002.html
4,TF_IDF,1,python programming language,4,en_articles_l_o_u_Lout_html,6.172097,"Lout - Wikipedia, the free encyclopedia",en/articles/l/o/u/Lout.html
